# EV-DeCAFS Component Tester

Run each cell in order to verify each module before executing the full pipeline.
Cells include PASS/FAIL assertions.

## Cell 1: Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import yaml

from src.visualization.style import apply_style
apply_style()
%matplotlib inline

with open('../config/params.yaml') as f:
    params = yaml.safe_load(f)

print('Config loaded.')
print('Datasets to run: bitcoin + welllog')

## Cell 2: Data Loading Test

In [ ]:
from src.data.loader import load_bitcoin_data, load_welllog_data

# --- Bitcoin ---
try:
    btc_train, btc_test, dates_train, dates_test = load_bitcoin_data(
        train_end_date=params['splitting']['bitcoin_train_end']
    )
    assert btc_train.ndim == 1, 'y_train should be 1-D'
    assert btc_test.ndim == 1, 'y_test should be 1-D'
    assert len(btc_train) > 0 and len(btc_test) > 0
    print(f'Bitcoin train: {btc_train.shape}, test: {btc_test.shape}')
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(btc_train[:500], lw=0.8)
    ax.set_title('Bitcoin log-price (first 500 train obs)')
    plt.show()
    print('PASS: Bitcoin data')
except NotImplementedError:
    print('SKIP: load_bitcoin_data not yet implemented')
except Exception as e:
    print(f'FAIL: {e}')

# --- Well-log ---
try:
    wl_train, wl_test, gt_cps, gt_outliers = load_welllog_data(
        train_fraction=params['splitting']['welllog_train_fraction']
    )
    assert wl_train.ndim == 1
    print(f'Well-log train: {wl_train.shape}, test: {wl_test.shape}')
    print(f'Known changepoints: {gt_cps}')
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(np.concatenate([wl_train, wl_test]), lw=0.7, color='gray')
    for cp in gt_cps:
        ax.axvline(cp, color='blue', lw=0.8, alpha=0.6)
    ax.set_title('Well-log signal with ground-truth changepoints')
    plt.show()
    print('PASS: Well-log data')
except NotImplementedError:
    print('SKIP: load_welllog_data not yet implemented')
except Exception as e:
    print(f'FAIL: {e}')

## Cell 3: AR(1) Parameter Estimation

In [ ]:
from src.phase1.ar1_model import estimate_ar1_params
from statsmodels.graphics.tsaplots import plot_acf

try:
    ar1 = estimate_ar1_params(btc_train)
    print(f"phi        = {ar1['phi']:.4f}")
    print(f"sigma_v^2  = {ar1['sigma_v_sq']:.6f}")
    print(f"sigma_eta^2= {ar1['sigma_eta_sq']:.6f}")
    assert abs(ar1['phi']) < 1, 'phi must satisfy stationarity: |phi| < 1'
    assert ar1['sigma_v_sq'] > 0
    assert ar1['sigma_eta_sq'] >= 1e-8

    # ACF of AR(1) residuals — should show no significant autocorrelation
    residuals = btc_train[1:] - ar1['phi'] * btc_train[:-1]
    fig, ax = plt.subplots(figsize=(10, 3))
    plot_acf(residuals, lags=40, ax=ax, title='ACF of AR(1) residuals (whiteness check)')
    plt.show()

    print('PASS: AR(1) estimation')
except NotImplementedError:
    print('SKIP: estimate_ar1_params not yet implemented')
except Exception as e:
    print(f'FAIL: {e}')

## Cell 4: EVT Penalty

In [ ]:
from src.phase1.evt_penalty import (
    compute_evi_field, compute_adaptive_penalty, compute_exceedance_count_penalty
)

try:
    p1 = params['phase1']
    xi = compute_evi_field(wl_train, w=p1['window_halfwidth_w'], q0=p1['gpd_percentile_q0'])
    assert xi.shape == wl_train.shape, 'xi must have same length as input'
    alpha_gpd = compute_adaptive_penalty(xi, p1['alpha_0'], p1['evt_sensitivity_lambda_ev'])
    assert (alpha_gpd >= p1['alpha_0']).all(), 'alpha_t must be >= alpha_0 everywhere'

    # Exceedance-count penalty (for comparison)
    from src.phase1.ar1_model import estimate_ar1_params as _ar1
    _ar = _ar1(wl_train)
    mu_proxy = np.full_like(wl_train, np.mean(wl_train))
    alpha_ec = compute_exceedance_count_penalty(
        wl_train, mu_proxy,
        sigma_v=float(np.sqrt(_ar['sigma_v_sq'])),
        w=p1['window_halfwidth_w'],
        c=p1['exceedance_multiplier_c'],
        alpha_0=p1['alpha_0'],
    )

    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    axes[0].plot(xi, lw=0.8)
    axes[0].set_title('EVI field xi_t')
    axes[1].plot(alpha_gpd, lw=0.8, label='GPD-based')
    axes[1].axhline(p1['alpha_0'], ls='--', color='red', label='alpha_0')
    axes[1].set_title('GPD adaptive penalty alpha_t')
    axes[1].legend()
    axes[2].plot(alpha_ec, lw=0.8, color='orange', label='Exceedance-count')
    axes[2].axhline(p1['alpha_0'], ls='--', color='red', label='alpha_0')
    axes[2].set_title('Exceedance-count penalty alpha_t')
    axes[2].legend()
    plt.tight_layout()
    plt.show()
    print('PASS: EVT penalty')
except NotImplementedError:
    print('SKIP: EVT penalty not yet implemented')
except Exception as e:
    print(f'FAIL: {e}')

## Cell 5: EV-DeCAFS

In [ ]:
import time
from src.phase1.decafs import ev_decafs

try:
    p1 = params['phase1']
    t0 = time.perf_counter()
    result = ev_decafs(
        wl_train, alpha_gpd,
        lambda_param=p1['lambda_param'],
        gamma=1.0,
        phi=ar1['phi'],
    )
    elapsed = time.perf_counter() - t0
    print(f'Detected {len(result["changepoints"])} changepoints in {elapsed:.2f}s')
    print(f'Total cost: {result["cost"]:.4f}')
    assert len(result['changepoints']) > 0, 'Expected at least one changepoint'
    assert result['means'].shape == wl_train.shape

    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(wl_train, lw=0.6, color='gray', label='data')
    ax.plot(result['means'], lw=1.5, color='red', label='mu_hat')
    for cp in result['changepoints']:
        ax.axvline(cp, color='black', lw=0.8, ls='--')
    ax.set_title('EV-DeCAFS on well-log training data')
    ax.legend()
    plt.show()
    print('PASS: EV-DeCAFS')
except NotImplementedError:
    print('SKIP: ev_decafs not yet implemented')
except Exception as e:
    print(f'FAIL: {e}')

## Cell 6: Feature Extraction

In [ ]:
from src.phase1.feature_extract import extract_features

try:
    X_train, feat_names = extract_features(
        wl_train,
        result['changepoints'],
        result['means'],
        L=params['labelling']['window_L'],
    )
    assert X_train.shape[1] == 4, f'Expected 4 features, got {X_train.shape[1]}'
    assert not np.any(np.isnan(X_train)), 'NaN in feature matrix'
    assert not np.any(np.isinf(X_train)), 'Inf in feature matrix'
    print(f'Feature matrix shape: {X_train.shape}')
    print(f'Features: {feat_names}')
    print(pd.DataFrame(X_train, columns=feat_names).describe())

    fig, axes = plt.subplots(1, 4, figsize=(14, 3))
    for i, (ax, name) in enumerate(zip(axes, feat_names)):
        ax.hist(X_train[:, i], bins=20)
        ax.set_title(name)
    plt.tight_layout()
    plt.show()
    print('PASS: Feature extraction')
except NotImplementedError:
    print('SKIP: extract_features not yet implemented')
except Exception as e:
    print(f'FAIL: {e}')

## Cell 7: Labelling

In [ ]:
from src.phase2.labelling import compute_kappa_mu, label_changepoints

try:
    lp = params['labelling']
    kappa_mu = compute_kappa_mu(X_train, percentile=lp['kappa_mu_percentile'])
    labels = label_changepoints(X_train, kappa_mu, kappa_S=lp['kappa_S'])
    assert labels.shape == (X_train.shape[0],)
    assert set(labels).issubset({0, 1})
    counts = {v: (labels == v).sum() for v in [0, 1]}
    print(f'kappa_mu = {kappa_mu:.4f}')
    print(f'Class distribution: {counts}')
    print('PASS: Labelling')
except NotImplementedError:
    print('SKIP: labelling not yet implemented')
except Exception as e:
    print(f'FAIL: {e}')

## Cell 8: SMOTE Balancing

In [ ]:
from src.phase2.smote_balance import balance_training_data

try:
    sp = params['smote']
    X_bal, y_bal = balance_training_data(
        X_train, labels,
        k_neighbors=sp['k_neighbors'],
        random_state=sp['random_state'],
    )
    pre = {v: (labels == v).sum() for v in [0, 1]}
    post = {v: (y_bal == v).sum() for v in [0, 1]}
    print(f'Before SMOTE: {pre}')
    print(f'After SMOTE:  {post}')
    assert post[0] == post[1], 'Classes should be balanced after SMOTE'

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, X, y, title in [
        (axes[0], X_train, labels, 'Before SMOTE'),
        (axes[1], X_bal, y_bal, 'After SMOTE'),
    ]:
        ax.scatter(X[y==0, 0], X[y==0, 1], alpha=0.5, label='recoiled')
        ax.scatter(X[y==1, 0], X[y==1, 1], alpha=0.5, label='sustained')
        ax.set_xlabel('delta_mu'); ax.set_ylabel('persistence')
        ax.set_title(title); ax.legend()
    plt.tight_layout(); plt.show()
    print('PASS: SMOTE')
except NotImplementedError:
    print('SKIP: SMOTE not yet implemented')
except Exception as e:
    print(f'FAIL: {e}')

## Cell 9: FPNN

In [ ]:
from src.phase2.fpnn import FourierPNN
from sklearn.metrics import classification_report, roc_curve, auc

try:
    fp = params['fpnn']
    fpnn = FourierPNN(J=fp['J_harmonics'], scaling_range=tuple(fp['scaling_range']))
    fpnn.fit(X_bal, y_bal)

    # Quick test on training data
    y_pred = fpnn.predict(X_bal)
    y_proba = fpnn.predict_proba(X_bal)
    assert y_pred.shape == (X_bal.shape[0],)
    assert y_proba.shape == (X_bal.shape[0], 2)
    assert np.allclose(y_proba.sum(axis=1), 1.0, atol=1e-6)

    print(classification_report(y_bal, y_pred, target_names=['recoiled', 'sustained']))
    fpr, tpr, _ = roc_curve(y_bal, y_proba[:, 1])
    print(f'AUC-ROC (train): {auc(fpr, tpr):.4f}')

    coefs = fpnn.get_coefficients()
    print(f'Coefficient keys: {list(coefs.keys())}')
    print('PASS: FPNN')
except NotImplementedError:
    print('SKIP: FourierPNN not yet implemented')
except Exception as e:
    print(f'FAIL: {e}')

## Cell 10: Baselines

In [ ]:
from src.phase2.baselines import get_baselines, train_and_evaluate_all

try:
    baselines = get_baselines(params['baselines'])
    print(f'Baselines: {list(baselines.keys())}')
    results_df = train_and_evaluate_all(
        X_bal, y_bal, X_train, labels, baselines, fpnn
    )
    print(results_df.to_string())
    assert 'Balanced Accuracy' in results_df.columns or len(results_df.columns) > 0
    print('PASS: Baselines')
except NotImplementedError:
    print('SKIP: baselines not yet implemented')
except Exception as e:
    print(f'FAIL: {e}')

## Cell 11: MRL Index

In [ ]:
from src.evaluation.mrl_index import (
    compute_mrl, compute_risk, compute_censored_mrl, compute_censored_risk
)

try:
    ev = params['evaluation']
    true_cp = int(gt_cps[0]) if len(gt_cps) > 0 else 400
    detected = result['changepoints']

    mrl_res = compute_mrl(detected, true_cp)
    print(f'FP={mrl_res["FP"]}, MRL={mrl_res["MRL"]}, T_first={mrl_res["T_first"]}')

    n_test = len(wl_test)
    Tmax = ev['censoring_Tmax_fraction'] * n_test
    eps = ev['censoring_epsilon']

    R = compute_risk(mrl_res['FP'], mrl_res['MRL'], cF=1, cD=1)
    R_tilde = compute_censored_risk(mrl_res['FP'], mrl_res['MRL'], 1, 1, eps, Tmax)
    print(f'R={R:.4f}, R_tilde={R_tilde:.4f}')

    # Edge cases
    assert compute_censored_mrl(0, eps, Tmax) == eps
    assert compute_censored_mrl(float('inf'), eps, Tmax) == Tmax
    print('PASS: MRL index')
except NotImplementedError:
    print('SKIP: MRL index not yet implemented')
except Exception as e:
    print(f'FAIL: {e}')

## Cell 12: Sensitivity Analysis

In [ ]:
from src.evaluation.sensitivity import cost_ratio_sensitivity
from src.visualization.sensitivity_heatmap import plot_sensitivity_heatmap

try:
    ev = params['evaluation']
    dummy_results = {
        'EV-DeCAFS': {'FP': mrl_res['FP'], 'MRL': mrl_res['MRL']},
        'PELT':      {'FP': 2, 'MRL': 10.0},
    }
    n_test = len(wl_test)
    Tmax = ev['censoring_Tmax_fraction'] * n_test
    rankings, raw = cost_ratio_sensitivity(
        dummy_results, ev['cost_cF_grid'], ev['cost_cD_grid'],
        epsilon=ev['censoring_epsilon'], Tmax=Tmax,
    )
    print(rankings)
    plot_sensitivity_heatmap(rankings, raw)
    print('PASS: Sensitivity analysis')
except NotImplementedError:
    print('SKIP: sensitivity not yet implemented')
except Exception as e:
    print(f'FAIL: {e}')

## Cell 13: Hausdorff Distance

In [ ]:
from src.evaluation.hausdorff import symmetric_hausdorff
from scipy.stats import spearmanr

try:
    detected = result['changepoints']
    h = symmetric_hausdorff(detected, gt_cps)
    print(f'Symmetric Hausdorff distance: {h:.2f}')
    assert h >= 0

    # Dummy correlation example with 2 detectors
    h_values = [h, h * 1.5]
    r_tilde = [R_tilde, R_tilde * 1.2]
    corr, pval = spearmanr(h_values, r_tilde)
    print(f'Spearman correlation (H, R_tilde) = {corr:.4f}, p = {pval:.4f}')
    print('PASS: Hausdorff distance')
except NotImplementedError:
    print('SKIP: Hausdorff not yet implemented')
except Exception as e:
    print(f'FAIL: {e}')

## Cell 14: Full Pipeline Dry Run

In [ ]:
import subprocess
import os
from pathlib import Path

result_proc = subprocess.run(
    ['python', '../scripts/run_pipeline.py', '--dataset', 'welllog', '--skip-baselines'],
    capture_output=True, text=True, timeout=600
)
print('STDOUT:', result_proc.stdout[-2000:] if result_proc.stdout else '(none)')
print('STDERR:', result_proc.stderr[-1000:] if result_proc.stderr else '(none)')
print(f'Return code: {result_proc.returncode}')

# Check output files
results_root = Path('../results')
figures = list((results_root / 'figures').glob('*'))
tables = list((results_root / 'tables').glob('*'))
print(f'Figures: {[f.name for f in figures]}')
print(f'Tables:  {[t.name for t in tables]}')

# Verify figure file sizes.  PDFs are vector so ~5 KB+ is fine;
# PNGs at 300 DPI should be > 100 KB.
for fig_path in figures:
    size_kb = os.path.getsize(fig_path) / 1024
    min_kb = 100 if fig_path.suffix.lower() == '.png' else 5
    print(f'  {fig_path.name}: {size_kb:.1f} KB')
    assert size_kb > min_kb, f'Figure {fig_path.name} too small ({size_kb:.1f} KB < {min_kb} KB)'